# 取引先資料：読み取り専用Google Sheets接続テスト

**目的**：専用サービスアカウント経由で、指定したシートの少数セルを読み、商品名や原価を表示せず件数・警告のみ表示します。

**実行前の条件**：
1. Google Cloudで、取引先解析専用のサービスアカウントを作成する（既存SP-API用アカウントを流用しない）。
2. Google Sheets APIとIAM Service Account Credentials APIを有効にする。
3. このノートブックでログインするGoogleアカウントに、専用サービスアカウント**単体**への Service Account Token Creator 権限を付与する。
4. 検証対象スプレッドシート1件を、専用サービスアカウントへ**閲覧者（Viewer）**として共有する。

秘密鍵ファイル、OAuthトークン、実見積や取引条件をNotebookやGitHubへ保存しないでください。プログラムはSheetsの読み取りAPIだけを使い、発注・Drive書き込み・Keepa API照合は行いません。

> ChatGPTのDrive連携の認証とは別です。この実行は、ユーザー自身がColabで認証操作を行う必要があります。

In [ ]:
# ① 公開コードを検証済みコミットに固定して取得
import os, sys, subprocess
PINNED_COMMIT = "48ab1e8b86f0b3dada34ae55b91c559b909b3150"
CODE_DIR = "/content/supplier-docs-readonly"
if not os.path.isdir(os.path.join(CODE_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--no-checkout", "https://github.com/gyallen0130/manufacturer-keepa-analysis.git", CODE_DIR],
        check=True, capture_output=True)
# Existing Colab runtimes may have cloned before this revision; fetch the latest branch.
subprocess.run(["git", "-C", CODE_DIR, "fetch", "--quiet", "origin", "feature/supplier-docs-normalizer-v1"],
               check=True, capture_output=True)
subprocess.run(["git", "-C", CODE_DIR, "checkout", "--detach", PINNED_COMMIT],
               check=True, capture_output=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", os.path.join(CODE_DIR, "manufacturer_checker", "requirements.txt")],
    check=True)
sys.path.insert(0, os.path.join(CODE_DIR, "manufacturer_checker"))
# Refresh any modules cached by an earlier run in this same runtime.
for module in tuple(sys.modules):
    if module == "supplier_docs" or module.startswith("supplier_docs."):
        del sys.modules[module]
print("読み取り専用試験コードを準備しました。コミット:", PINNED_COMMIT[:12])

In [ ]:
# ② 自分のGoogleアカウントで認証（CloudのToken Creator権限が必要）
from google.colab import auth
auth.authenticate_user()
print("Googleアカウントのログインが完了しました。")

In [ ]:
# ③ 接続する専用サービスアカウントと対象Sheetを入力（Sheet IDは非表示入力）
import getpass
from supplier_docs.cli import main

service_account = input("取引先解析専用サービスアカウントのメール: ").strip()
sheet_id = getpass.getpass("テスト対象のGoogle Sheets ID（画面に表示しません）: ").strip()
sheet_tab = input("シート名 [見積もり]: ").strip() or "見積もり"
tab_escaped = sheet_tab.replace("'", "''")
a1_range = "'" + tab_escaped + "'!A1:L40"

try:
    main([
        "--sheets-id", sheet_id,
        "--sheet-range", a1_range,
        "--document-type", "quote_request",
        "--impersonate-service-account", service_account,
        "--summary-only",
    ])
except Exception as exc:
    # Google API exceptions can contain document URLs / account information: do not print raw exceptions.
    print("接続・解析に失敗しました。エラー種別:", type(exc).__name__)
    print("権限・APIの有効化・シート名を確認してください。秘密情報やスタックトレースは共有しないでください。")

## 成功した場合

表示されるのは `record_count`、`risk_counts`、`warning_code_counts`、`summary.inferred_fields`（推定された項目名）の集計だけです。見出しのない列を推定しても、ASINの正しさや利益額を確認したことにはなりません。Sheet ID・取引先の名前・JAN・ASIN・卸値は表示しません。

- 失敗してもGoogle Driveの原本は変更されません。
- 最初の成功後も、別シートや資料の全件自動収集は未実装です。
- 実行後はColabのセッションを切断し、Notebookの入力・出力に識別情報が残っていないか確認してください。
- GitHubへ認証情報・取引先実データ・実行結果のスクリーンショットをアップしないでください。

コード・確認手順については `manufacturer_checker/supplier_docs/README.md` を参照してください。